In [ ]:
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sparse
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.svm import LinearSVC


# ============================================================================
# 07. UNMASKED (FULL-TEXT) SVM BASELINE
# ============================================================================
# Mirrors 03 (feature extraction) + 04 (SVM training) but on the ORIGINAL
# unmasked text instead of the POS-masked text, so there is a full-text
# baseline to contrast against the masked-only results from 03/04.
# Without this notebook there is no way to measure how much SVM accuracy
# comes from content/lexical cues vs. structural cues -- the same question
# 09 asks of the LLM.

BASE_DIR = Path("..").resolve()
INPUT_CSV_PATH = BASE_DIR / "data/processed/combined_nli_masked.csv"  # has both text + masked_text

FEATURES_DIR = BASE_DIR / "data/processed/features_fulltext"
MATRICES_PATH = BASE_DIR / "data/processed/nli_feature_matrices_fulltext.pkl"
MODELS_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "data/processed/results"

MODEL_PATH = MODELS_DIR / "linear_svc_nli_model_fulltext.pkl"
PREDICTIONS_PATH = RESULTS_DIR / "svm_predictions_fulltext.csv"
SUMMARY_PATH = RESULTS_DIR / "svm_metrics_fulltext.csv"

FEATURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Loading combined corpus...")
df = pd.read_csv(INPUT_CSV_PATH)
df["text"] = df["text"].fillna("").astype(str)
df["label"] = df["label"].astype(str).str.strip().str.upper()
df["dataset"] = df["dataset"].astype(str).str.strip().str.upper()
df["split"] = df["split"].astype(str).str.strip().str.lower()

train_mask = ((df["dataset"] == "TOEFL11") & (df["split"] == "train")).to_numpy()
toefl_test_mask = ((df["dataset"] == "TOEFL11") & (df["split"] == "test")).to_numpy()
icle_test_mask = (df["dataset"] == "ICLEV3").to_numpy()

train_texts = df.loc[train_mask, "text"]
if train_texts.empty:
    raise ValueError("No TOEFL11 train rows found. Cannot fit vectorizers.")

print(f"Fitting vectorizers on {len(train_texts)} TOEFL11 training essays (unmasked text)...")

# Same feature configuration as 03_feature_extraction.ipynb so the
# masked-vs-unmasked SVM comparison isolates the effect of masking, not a
# change in feature design.
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 3),
    max_features=5000,
    min_df=2,
    sublinear_tf=True,
    norm="l2",
)
char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    max_features=10000,
    min_df=2,
    sublinear_tf=True,
    norm="l2",
)

word_vectorizer.fit(train_texts)
char_vectorizer.fit(train_texts)

print("Transforming all documents...")
X_word = word_vectorizer.transform(df["text"])
X_char = char_vectorizer.transform(df["text"])
X = sparse.hstack([X_word, X_char], format="csr")
y = df["label"].to_numpy()
doc_id = df["doc_id"].to_numpy()
dataset_tags = df["dataset"].to_numpy()
split_tags = df["split"].to_numpy()

with (FEATURES_DIR / "word_vectorizer_fulltext.pkl").open("wb") as f:
    pickle.dump(word_vectorizer, f)
with (FEATURES_DIR / "char_vectorizer_fulltext.pkl").open("wb") as f:
    pickle.dump(char_vectorizer, f)
with MATRICES_PATH.open("wb") as f:
    pickle.dump(
        {
            "X": X,
            "y": y,
            "doc_id": doc_id,
            "dataset_tags": dataset_tags,
            "split_tags": split_tags,
            "feature_blocks": {"word": X_word.shape[1], "char": X_char.shape[1]},
        },
        f,
    )

print(f"Feature matrix shape: {X.shape}")

X_train, y_train = X[train_mask], y[train_mask]
print(f"Training rows: {X_train.shape[0]}")

model = LinearSVC(C=1.0, random_state=42, max_iter=10000)

with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always", ConvergenceWarning)
    model.fit(X_train, y_train)

convergence_warnings = [w for w in caught_warnings if issubclass(w.category, ConvergenceWarning)]
if convergence_warnings:
    print("Warning: LinearSVC emitted a convergence warning. Consider increasing max_iter or tuning C.")


def evaluate_partition(mask: np.ndarray, partition_name: str) -> tuple[pd.DataFrame, dict]:
    X_eval = X[mask]
    y_true = y[mask]
    y_pred = model.predict(X_eval)

    pred_df = pd.DataFrame(
        {
            "doc_id": doc_id[mask],
            "dataset": dataset_tags[mask],
            "split": split_tags[mask],
            "true_l1": y_true,
            "svm_predicted_l1": y_pred,
            "partition": partition_name,
        }
    )

    metrics = {
        "partition": partition_name,
        "n": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }

    print(f"\n=== {partition_name} (unmasked/full-text SVM) ===")
    print(pd.Series(metrics))
    print(classification_report(y_true, y_pred, digits=4))

    labels = sorted(np.unique(y))
    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=labels),
        index=[f"true_{x}" for x in labels],
        columns=[f"pred_{x}" for x in labels],
    )
    cm.to_csv(RESULTS_DIR / f"svm_confusion_fulltext_{partition_name}.csv")

    return pred_df, metrics


toefl_predictions, toefl_metrics = evaluate_partition(toefl_test_mask, "toefl11_test")
icle_predictions, icle_metrics = evaluate_partition(icle_test_mask, "iclev3_external_test")

all_predictions = pd.concat([toefl_predictions, icle_predictions], ignore_index=True)
all_predictions.to_csv(PREDICTIONS_PATH, index=False)
pd.DataFrame([toefl_metrics, icle_metrics]).to_csv(SUMMARY_PATH, index=False)

with MODEL_PATH.open("wb") as f:
    pickle.dump(model, f)

print(f"Saved model: {MODEL_PATH}")
print(f"Saved predictions: {PREDICTIONS_PATH}")
print(f"Saved metrics: {SUMMARY_PATH}")

# Quick contrast against the masked-text SVM from 04, if available.
masked_summary_path = RESULTS_DIR / "svm_metrics_full.csv"
if masked_summary_path.exists():
    masked_summary = pd.read_csv(masked_summary_path)
    fulltext_summary = pd.DataFrame([toefl_metrics, icle_metrics])
    merged = masked_summary.merge(fulltext_summary, on="partition", suffixes=("_masked", "_fulltext"))
    merged["shortcut_delta_accuracy"] = merged["accuracy_fulltext"] - merged["accuracy_masked"]
    print("\n=== SVM masked vs. full-text accuracy delta ===")
    print(merged[["partition", "accuracy_masked", "accuracy_fulltext", "shortcut_delta_accuracy"]])
else:
    print("\nRun 04_model_training.ipynb first to compare against the masked-text SVM baseline.")
